# Object Detection + Segmentação: pipeline em 1h

Versão reduzida, para pós-graduação, do assignment de graduação **Object Detection + Semantic Segmentation** (`deep-learning-course-fgv/tarefas/Object_Detection.ipynb`).

O assignment original pede implementar a YOLOv3 do zero, treinar uma U-Net própria em patches, medir mAP/AP formalmente e fazer fine-tuning do Mask R-CNN — um projeto de várias semanas. Aqui mantemos a ideia central (**detectar → recortar → segmentar → costurar → comparar com um modelo ponta-a-ponta**), trocando "treinar tudo do zero" por modelos **pré-treinados**, o que cabe em cerca de 1h.

**Roteiro:**

| Duração | Bloco |
|---|---|
| 5 min | Setup |
| 10 min | Detecção com YOLO pré-treinada |
| 15 min | Segmentação por patch com modelo pré-treinado |
| 10 min | Costurar de volta e visualizar |
| 10 min | Comparação com Mask R-CNN ponta-a-ponta |
| 5 min | Discussão e encerramento |

*(A "Trilha avançada" no fim do notebook — fine-tuning real no VOC, SAM, métricas formais — é opcional e não faz parte do núcleo cronometrado.)*

In [ ]:
%pip install ultralytics -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
from torchvision.transforms.functional import to_tensor

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# Imagem de teste: a mesma usada nos exemplos oficiais da Ultralytics (tem pessoas + ônibus, boa para o pipeline).
# Troque por uma foto sua se quiser (uma URL ou um caminho local funcionam).
IMG_URL = "https://ultralytics.com/images/bus.jpg"

## 1. Detecção com YOLO pré-treinada (10 min)

Em vez de implementar e treinar a YOLOv3 do zero (como pede o assignment original), usamos a YOLOv8n já pré-treinada na COCO. Isso já basta para detectar bem várias classes — vamos ver por quê na próxima célula.

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')  # pesos pré-treinados na COCO, baixados automaticamente na primeira execução
results = yolo.predict(source=IMG_URL, conf=0.4)
result = results[0]

# O ultralytics carrega a imagem com OpenCV, que usa BGR — convertendo para RGB antes de qualquer plot
img_rgb = result.orig_img[..., ::-1]

boxes = result.boxes.xyxy.cpu().numpy()
classes = result.boxes.cls.cpu().numpy().astype(int)
scores = result.boxes.conf.cpu().numpy()

print(f"{len(boxes)} objetos detectados: {[result.names[c] for c in classes]}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_rgb)

for box, cls, score in zip(boxes, classes, scores):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f"{result.names[cls]}: {score:.2f}", color='white', fontsize=9, backgroundcolor='green')

ax.axis('off')
ax.set_title('Detecção com YOLOv8n pré-treinada (zero-shot, sem fine-tuning)')
plt.show()

**Por que isso já funciona sem treinar nada?** A COCO (dataset em que a YOLOv8n foi pré-treinada) e o PASCAL VOC (dataset do assignment original) têm bastante sobreposição de classes — pessoa, ônibus/carro, cachorro, gato... Por isso a detecção "de fábrica" já é útil. Quem quiser o fine-tuning real no VOC (o que o assignment pede) encontra o código na **Trilha avançada**, no fim do notebook.

## 2. Segmentação por patch com modelo pré-treinado (15 min)

Para cada objeto detectado, vamos recortar a região da caixa e passar por um segmentador semântico **pré-treinado** — em vez de treinar uma U-Net própria do zero em patches, como pede o assignment original.

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

# Essa variante foi treinada numa versão da COCO rotulada com as 21 classes do VOC (20 classes + fundo) —
# o mesmo esquema de classes do dataset do assignment original.
seg_weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
seg_model = deeplabv3_resnet50(weights=seg_weights).to(device).eval()
seg_preprocess = seg_weights.transforms()
voc_classes = seg_weights.meta["categories"]
print(voc_classes)

In [ ]:
def segmentar_patch(patch_rgb):
    """Recebe um patch (numpy array HxWx3, RGB) e devolve uma máscara booleana do mesmo tamanho:
    True nos pixels que o modelo classificou como algum objeto (isto é, não-fundo)."""
    patch_pil = Image.fromarray(patch_rgb)
    input_tensor = seg_preprocess(patch_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = seg_model(input_tensor)["out"][0]

    pred_classes = output.argmax(0).byte().cpu().numpy()  # classe por pixel, na resolução do preprocess
    pred_img = Image.fromarray(pred_classes).resize(patch_pil.size, resample=Image.NEAREST)  # volta ao tamanho original do patch
    return np.array(pred_img) != 0  # 0 é a classe "fundo" (__background__)

**Simplificação:** em vez de checar se a classe prevista pelo segmentador bate com a classe que o YOLO detectou, tratamos qualquer pixel não-fundo dentro do patch como parte do objeto. Isso é suficiente para ver o pipeline funcionando — um sistema de produção validaria também a classe.

In [ ]:
patch_masks = []

for box in boxes:
    x1, y1, x2, y2 = box.astype(int)
    patch = img_rgb[y1:y2, x1:x2]
    mask = segmentar_patch(patch)
    patch_masks.append(mask)

# Visualizar a máscara de cada patch
fig, axs = plt.subplots(1, len(boxes), figsize=(4 * len(boxes), 4))
if len(boxes) == 1:
    axs = [axs]

for ax, box, mask, cls in zip(axs, boxes, patch_masks, classes):
    x1, y1, x2, y2 = box.astype(int)
    ax.imshow(img_rgb[y1:y2, x1:x2])
    ax.imshow(mask, alpha=0.5, cmap='Reds')
    ax.set_title(result.names[cls])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Costurar de volta e visualizar (10 min)

Agora que temos uma máscara por objeto (do tamanho do patch), vamos colar cada uma de volta na posição original — a caixa delimitadora — numa máscara do tamanho da imagem inteira.

**Complete a linha que falta:** dado `x1, y1, x2, y2` de uma caixa e a `mask` do patch correspondente, em que região de `full_mask` (que tem o tamanho da imagem original) essa máscara deveria ser colada?

In [ ]:
full_mask = np.zeros(img_rgb.shape[:2], dtype=bool)

for box, mask in zip(boxes, patch_masks):
    x1, y1, x2, y2 = box.astype(int)
    ## seu código aqui ##

# Overlay vermelho semi-transparente sobre a imagem original
overlay = img_rgb.copy()
overlay[full_mask] = (0.4 * overlay[full_mask] + 0.6 * np.array([255, 60, 60])).astype(np.uint8)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(overlay)
for box, cls in zip(boxes, classes):
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none'))
    ax.text(x1, y1 - 5, result.names[cls], color='white', fontsize=9, backgroundcolor='green')
ax.axis('off')
ax.set_title('Pipeline de 2 estágios: detectar → recortar → segmentar → costurar')
plt.show()

#### resposta

In [ ]:
# full_mask[y1:y2, x1:x2] |= mask
#
# (o "|=" importa se duas caixas se sobrepõem: sem ele, colar a segunda máscara
# apagaria pixels que a primeira já tinha marcado como objeto)

## 4. Comparação com Mask R-CNN ponta-a-ponta (10 min)

Em vez de fazer fine-tuning do Mask R-CNN (como pede o assignment original), usamos direto os pesos pré-treinados na COCO e comparamos visualmente com o pipeline de 2 estágios acima, na mesma imagem.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

mrcnn_weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mrcnn = maskrcnn_resnet50_fpn(weights=mrcnn_weights).to(device).eval()
mrcnn_preprocess = mrcnn_weights.transforms()

img_tensor = mrcnn_preprocess(to_tensor(img_rgb)).to(device)

with torch.no_grad():
    pred = mrcnn([img_tensor])[0]

CONF_THRESHOLD = 0.7
high_conf = pred["scores"] > CONF_THRESHOLD
mrcnn_masks = (pred["masks"][high_conf] > 0.5).squeeze(1)  # (N, H, W) booleano
mrcnn_combined_mask = mrcnn_masks.any(dim=0).cpu().numpy()
mrcnn_labels = [mrcnn_weights.meta["categories"][i] for i in pred["labels"][high_conf].cpu().numpy()]

print(f"Mask R-CNN encontrou {high_conf.sum().item()} objetos: {mrcnn_labels}")

In [ ]:
overlay_2stage = img_rgb.copy()
overlay_2stage[full_mask] = (0.4 * overlay_2stage[full_mask] + 0.6 * np.array([255, 60, 60])).astype(np.uint8)

overlay_mrcnn = img_rgb.copy()
overlay_mrcnn[mrcnn_combined_mask] = (0.4 * overlay_mrcnn[mrcnn_combined_mask] + 0.6 * np.array([60, 120, 255])).astype(np.uint8)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(overlay_2stage)
axs[0].set_title('Pipeline de 2 estágios (YOLO + DeepLabV3)')
axs[0].axis('off')

axs[1].imshow(overlay_mrcnn)
axs[1].set_title('Ponta-a-ponta (Mask R-CNN)')
axs[1].axis('off')

plt.tight_layout()
plt.show()

## 5. Discussão e encerramento (5 min)

- Onde os dois pipelines discordam? Em quais objetos um acerta e o outro erra?
- O pipeline de 2 estágios herda os erros do detector: o que acontece com a segmentação quando a caixa do YOLO é imprecisa (corta parte do objeto, por exemplo)?
- O que aconteceria com uma classe que não existe na COCO (ex: uma peça industrial específica, um tipo de fruta local)? Qual dos dois pipelines seria mais fácil de adaptar para essa classe nova?
- Compare a quantidade de "peças móveis" de cada abordagem (dois modelos + lógica de recorte/costura vs. um modelo só). O que isso te diz sobre a troca entre modularidade (2 estágios) e simplicidade de manutenção (ponta-a-ponta)?

Deixe seu feedback da aula [aqui](https://forms.gle/DSZsGDpkaGi3f2FW7) por favor :)

## Trilha avançada (opcional)

O assignment original da graduação (`deep-learning-course-fgv/tarefas/Object_Detection.ipynb`) pede a versão completa: implementar a YOLOv3 do zero, treinar uma U-Net própria em patches, medir mAP/AP formalmente e fazer fine-tuning do Mask R-CNN. As três opções abaixo vão nessa direção — nenhuma é necessária para completar a aula, e nenhuma delas roda "out of the box" (todas exigem algo a mais: dataset, download extra ou anotações de referência).

### Opção A: fine-tuning de verdade da YOLO no PASCAL VOC

Requer baixar o [dataset do Kaggle](https://www.kaggle.com/datasets/gopalbhattrai/pascal-voc-2012-dataset). O código completo de conversão VOC→YOLO e treino já existe e funciona — está em `deep-learning-course-fgv/tarefas/Object_Detection.ipynb`. Resumo do formato de conversão:

In [ ]:
# import xml.etree.ElementTree as ET
#
# label_dict = {
#     'aeroplane': 0, 'bicycle': 1, 'bird': 2, 'boat': 3, 'bottle': 4,
#     'bus': 5, 'car': 6, 'cat': 7, 'chair': 8, 'cow': 9,
#     'diningtable': 10, 'dog': 11, 'horse': 12, 'motorbike': 13, 'person': 14,
#     'pottedplant': 15, 'sheep': 16, 'sofa': 17, 'train': 18, 'tvmonitor': 19
# }
#
# def create_yolo_annotation(xml_file_path, yolo_label_path, label_dict):
#     # PASCAL VOC (xml, cantos xmin/ymin/xmax/ymax) -> formato YOLO (classe x_center y_center largura altura, normalizados)
#     tree = ET.parse(xml_file_path)
#     root = tree.getroot()
#     annotations = []
#     img_width = int(root.find('size/width').text)
#     img_height = int(root.find('size/height').text)
#     for obj in root.findall('object'):
#         label = obj.find('name').text
#         if label not in label_dict:
#             continue
#         bndbox = obj.find('bndbox')
#         xmin, ymin = float(bndbox.find('xmin').text), float(bndbox.find('ymin').text)
#         xmax, ymax = float(bndbox.find('xmax').text), float(bndbox.find('ymax').text)
#         x_center = ((xmin + xmax) / 2) / img_width
#         y_center = ((ymin + ymax) / 2) / img_height
#         width = (xmax - xmin) / img_width
#         height = (ymax - ymin) / img_height
#         annotations.append(f"{label_dict[label]} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
#     with open(yolo_label_path, 'w') as f:
#         f.write("\n".join(annotations))
#
# # depois de converter todas as anotações (ver notebook original para o loop completo):
# # model = YOLO('yolov8n.pt')
# # model.train(data='yolo_dataset/data.yaml', epochs=2, imgsz=640, batch=16)

### Opção B: trocar o segmentador por SAM (Segment Anything)

Em vez do DeepLabV3, o SAM usa a própria caixa do YOLO como *prompt* — é conceitualmente mais direto para esse pipeline ("aqui está uma caixa, me dá a máscara exata"), e a máscara já sai no tamanho da imagem original (sem precisar do passo de "costurar"). Veja `deep-learning-course-fgv/tarefas/SAM.ipynb` para o assignment completo de fine-tuning. Esqueleto de uso (não incluído nesta aula por exigir um download extra de ~375MB):

In [ ]:
# %pip install segment-anything -q
# from segment_anything import sam_model_registry, SamPredictor
#
# sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth").to(device)
# predictor = SamPredictor(sam)
# predictor.set_image(img_rgb)
#
# for box in boxes:
#     mask, score, _ = predictor.predict(box=box, multimask_output=False)
#     # mask[0] já vem no tamanho da imagem original — sem recortar/costurar!

### Opção C: medir mAP/AP de verdade

Para ir além da comparação visual e medir formalmente, como pede o assignment original — precisa de caixas *ground-truth* (ex: do VOC) para comparar:

In [ ]:
# %pip install torchmetrics -q
# from torchmetrics.detection.mean_ap import MeanAveragePrecision
#
# metric = MeanAveragePrecision()
# metric.update(
#     preds=[{"boxes": torch.tensor(boxes), "scores": torch.tensor(scores), "labels": torch.tensor(classes)}],
#     target=[{"boxes": ground_truth_boxes, "labels": ground_truth_labels}],  # do VOC, não temos nesta aula
# )
# print(metric.compute())

## Referências

- Assignment completo (versão graduação): `deep-learning-course-fgv/tarefas/Object_Detection.ipynb`
- [Documentação da Ultralytics YOLO](https://docs.ultralytics.com/)
- [torchvision.models.segmentation](https://docs.pytorch.org/vision/stable/models.html#semantic-segmentation)
- [torchvision.models.detection](https://docs.pytorch.org/vision/stable/models.html#instance-segmentation)